### Reading tables from bronze layer and performing Transformation

In [0]:
df = spark.table('data_lakehouse.bronze.crm_cust_info')

In [0]:
df.display()

### Removing extra spaces from the string columns

In [0]:
from pyspark.sql.functions import *

df2 = df.withColumn('cst_firstname', trim(col('cst_firstname')))

df2.display()

In [0]:
df2.withColumn('cst_lastname',trim(col('cst_lastname'))).display()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

for field in df.schema.fields:
    if isinstance(field.dataType, StringType) :
        df = df.withColumn(field.name, trim(field.name))

display(df)

### Normaliazation of values from column

In [0]:
from pyspark.sql.functions import *

df = (
    df
    .withColumn(
            'cst_marital_status',
            when(upper(col('cst_marital_status')) == 'M', 'Married')
            .when(upper(col('cst_marital_status')) == 'S','Single')
            .otherwise('N/A')              
               )
   
    .withColumn(
                'cst_gndr', 
                when(upper(col('cst_gndr')) == 'M', 'Male')
                .when(upper(col('cst_gndr')) == 'F','Female')
                .otherwise('N/A')
        )
)
                   
    
df.display()

### Renaming the columns 

In [0]:
RENAMEED_COLUMNS = {

    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_date'

}

In [0]:
for old_name,new_name in RENAMEED_COLUMNS.items():
    df = df.withColumnRenamed(old_name, new_name)


df.display()

### Writing the cleaned data to Silver Layer

In [0]:
from pyspark.sql.functions import *

df.write.mode('overwrite').format('delta').saveAsTable('data_lakehouse.silver.crm_cust_info')

In [0]:
df = spark.read.table('data_lakehouse.silver.crm_cust_info')

df.display()